# 第9章 クラス

このノートブックでは **クラス** を学びます。
データ分析でも `LinearRegression()` などのモデルはクラスのインスタンス。

## 学習目標
- クラスの定義とインスタンス化ができる
- `__init__` と `self` の役割を理解する
- インスタンス変数とメソッドを使える
- 継承の概念を理解する
- 特殊メソッド (`__str__`, `__repr__`) を使える


## 9.1 クラスとは

クラスは **「データ + 操作」をひとまとめにした設計図**。
クラスから作られた実体を **インスタンス (オブジェクト)** といいます。

例) "学生" を表したい場合
- データ: 名前, 学年, 点数
- 操作: 平均を計算する, 合格判定をする


## 9.2 クラスの定義

```python
class クラス名:
    def __init__(self, 引数1, 引数2, ...):
        self.属性1 = 引数1
        self.属性2 = 引数2

    def メソッド名(self, 引数):
        # self を使ってインスタンス自身の属性にアクセス
        return ...
```

- `__init__` はインスタンス生成時に **自動で呼ばれる初期化メソッド** (コンストラクタ)
- `self` は **そのインスタンス自身** を指す (慣習でこの名前)
- メソッドの第 1 引数は必ず `self` を書く (呼び出し時には書かない)


In [ ]:
class Student:
    def __init__(self, name, grade):
        self.name = name        # 属性 name に保存
        self.grade = grade      # 属性 grade に保存
        self.scores = []        # 空のリストで初期化

    def add_score(self, subject, score):
        self.scores.append((subject, score))

    def average(self):
        if not self.scores:
            return 0
        total = sum(s for _, s in self.scores)
        return total / len(self.scores)


## 9.3 インスタンスの作成と使い方

クラス名を関数のように呼ぶとインスタンスが作られます。
このとき `__init__` の引数を渡します (`self` は渡さない)。


In [ ]:
# インスタンス化 (Student('田中', 2) で __init__ が呼ばれる)
s = Student('田中', 2)

print(s.name)       # '田中'
print(s.grade)      # 2
print(s.scores)     # []

# メソッドを呼ぶ
s.add_score('国語', 80)
s.add_score('数学', 90)
s.add_score('英語', 75)

print(s.scores)        # [('国語', 80), ('数学', 90), ('英語', 75)]
print(s.average())     # 81.66...


### `self` がしていること

メソッドを `s.add_score('国語', 80)` のように呼ぶと、Python は内部的に
`Student.add_score(s, '国語', 80)` のように **インスタンス自身を第1引数に渡し直し** ています。
そのため、メソッド側の第 1 引数 `self` には `s` 自身が入ります。


## 9.4 複数のインスタンス

同じクラスから何個でもインスタンスを作れます。
各インスタンスは **独立した属性** を持ちます。


In [ ]:
s1 = Student('田中', 2)
s2 = Student('佐藤', 3)

s1.add_score('数学', 90)
s2.add_score('数学', 70)
s2.add_score('英語', 85)

print(s1.name, s1.average())
print(s2.name, s2.average())

# それぞれのインスタンスは独立 (s1 を変えても s2 は変わらない)


## 9.5 特殊メソッド (ダンダーメソッド)

`__` で始まり `__` で終わるメソッドは、Python 側から自動的に呼ばれる特別なもの。


In [ ]:
class Student:
    def __init__(self, name, grade):
        self.name = name
        self.grade = grade

    def __str__(self):
        """print() で表示されるときの文字列"""
        return f'{self.name} (学年: {self.grade})'

    def __repr__(self):
        """開発者向けの分かりやすい表現 (リスト等に入れたときに使われる)"""
        return f'Student({self.name!r}, {self.grade})'

s = Student('田中', 2)
print(s)              # __str__ が呼ばれる
print([s, s])         # __repr__ が呼ばれる


## 9.6 継承

既存のクラスを **基にして** 新しいクラスを作れます。
共通部分を親クラスにまとめ、差分だけ子クラスに書く設計。


In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def cry(self):
        return '(無音)'


class Dog(Animal):       # Animal を継承
    def cry(self):       # メソッドを上書き (オーバーライド)
        return 'ワン!'


class Cat(Animal):
    def cry(self):
        return 'ニャー'


for a in [Dog('ポチ'), Cat('タマ'), Animal('???')]:
    print(a.name, a.cry())


## 9.7 クラス変数とインスタンス変数

- **インスタンス変数**: `self.x = ...` で代入する変数。インスタンスごとに別物。
- **クラス変数**: クラス本体で `x = ...` と書く変数。**全インスタンスで共有**。


In [ ]:
class Counter:
    total = 0                   # クラス変数 (全インスタンス共有)

    def __init__(self, name):
        self.name = name        # インスタンス変数 (個別)
        Counter.total += 1

c1 = Counter('A')
c2 = Counter('B')
c3 = Counter('C')

print(Counter.total)    # 3
print(c1.name, c2.name, c3.name)


## 9.8 データ分析でのクラスの使い方の例

scikit-learn のモデルはクラスのインスタンス。`fit` / `predict` メソッドを持つのが定番。


In [ ]:
# 自作の「最小2乗法による直線フィッタ」のサンプル
class SimpleLinearFitter:
    """y = a*x + b の直線を fit する単純なクラス"""

    def __init__(self):
        self.a = None
        self.b = None

    def fit(self, xs, ys):
        n = len(xs)
        mean_x = sum(xs) / n
        mean_y = sum(ys) / n
        num = sum((x - mean_x) * (y - mean_y) for x, y in zip(xs, ys))
        den = sum((x - mean_x) ** 2 for x in xs)
        self.a = num / den
        self.b = mean_y - self.a * mean_x
        return self

    def predict(self, xs):
        return [self.a * x + self.b for x in xs]


model = SimpleLinearFitter()
model.fit([1, 2, 3, 4, 5], [2, 4, 6, 8, 10])
print(f'y = {model.a} * x + {model.b}')
print(model.predict([6, 7]))


## 9.9 練習問題


In [ ]:
# Q1. 何が表示される?
class Box:
    def __init__(self, w, h, d):
        self.w = w
        self.h = h
        self.d = d

    def volume(self):
        return self.w * self.h * self.d

b = Box(2, 3, 4)
print(b.volume())


**Q1 解説**

- `__init__` で `w=2, h=3, d=4`
- `volume()` は `2 * 3 * 4 = 24`


In [ ]:
# Q2. 出力は?
class Counter:
    def __init__(self):
        self.value = 0

    def add(self, n):
        self.value += n
        return self.value

c = Counter()
c.add(5)
c.add(10)
print(c.add(2))


**Q2 解説**

- 初期 `value = 0`
- `add(5)` → `value = 5`, 戻り値 5
- `add(10)` → `value = 15`, 戻り値 15
- `add(2)` → `value = 17`, 戻り値 17 → これが表示


## まとめ

- `class クラス名:` で設計図を定義
- `__init__(self, ...)` でインスタンス生成時の初期化
- `self` はそのインスタンス自身を指す
- メソッド内では `self.属性` で属性にアクセス
- 継承で既存クラスを拡張できる
- `__str__`, `__repr__` などの特殊メソッドで挙動をカスタマイズ
